# UMT-ViT — Universal Multi-Scale Topographic Vision Transformer

**Self-supervised, dataset-agnostic research notebook** · design contract:
[`docs/UMT-VIT-ARCHITECTURE.md`](https://github.com/LarsGroep/DragonHatchling/blob/claude/umt-vit-opus-orchestration-zpd03a/docs/UMT-VIT-ARCHITECTURE.md) ·
research record: [`docs/UMT-VIT-RESEARCH.md`](https://github.com/LarsGroep/DragonHatchling/blob/claude/umt-vit-opus-orchestration-zpd03a/docs/UMT-VIT-RESEARCH.md)

```
image ─▶ dual-scale patches (fine + coarse) ─▶ cross-scale attention
      ─▶ fusion ─▶ transformer encoder (ALL layers kept)
      ─▶ spatial uplifting: layer l → depth slice z=l
      ─▶ 3-D latent voxel volume  V ∈ R^{H'×W'×L×C}
      ─▶ differentiable 3-D Self-Organizing Map (topology-preserving)
      losses: NT-Xent · SOM quantization · smoothness · layer-scale
              ordering (geodesic: ablation-gated, off by default)
```

**Honesty notes** (from the research record):
- The Z-axis is a *learned hierarchy of representations* (transformer depth),
  never physical/anatomical depth. ViT layers do **not** order themselves by
  spatial scale on their own (Raghu et al., NeurIPS 2021) — we *impose* that
  bias with an ordering regularizer and then **measure** whether it emerged
  (frequency probe near the end of this notebook).
- The SOM is a differentiable soft-SOM (DESOM/DPSOM lineage). The literal
  Kohonen/Hebbian EMA update is available as `model.som_update="kohonen_ema"`
  for the biological-variant ablation.
- No labels are used for representation learning. Labels, when present, are
  used only by the evaluation section (linear probe / k-NN).

## How to swap datasets

Edit **only the CONFIG cell below** — pick a preset or write your own
`dataset` block. Everything downstream (model, losses, training,
visualisations, animations, evaluation, export) derives from it.

| you have | set `loader` | notes |
|---|---|---|
| nothing (demo/CI) | `"shapes"` | generated on the fly, zero downloads |
| `root/<class>/*.jpg` | `"imagefolder"` | optional `train/ val/ test/` subtrees honored |
| a metadata CSV + image dir(s) | `"csv"` | supports label / group columns (leakage-free grouped splits) |

Ready-made presets: `"shapes"` (default, runs on CPU in minutes),
`"ham10000"` (dermoscopy — attach the Kaggle dataset
`kmader/skin-cancer-mnist-ham10000`), `"eurosat"` (satellite — attach
`apollo2506/eurosat-dataset`). Unlabeled data: set `label_column: None`;
SSL training is unaffected and evaluation degrades gracefully.

In [ ]:
# ════════════════ CONFIG — the only cell you need to edit ════════════════
import copy

CONFIG = {
    "dataset": {
        "name": "shapes",            # free-form tag used in titles/exports
        "loader": "shapes",          # shapes | imagefolder | csv
        "image_dir": None,           # imagefolder root · csv: dir or list of dirs
        "metadata_csv": None,        # csv loader only
        "path_column": "image_id",   # csv: column holding the file stem/name
        "path_suffix": ".jpg",       # csv: appended to path_column values
        "label_column": "shape",     # None => fully unlabeled mode
        "group_column": None,        # e.g. "lesion_id" => leakage-free splits
        "image_size": 64,
        "channels": 3,
        "n_per_class": 90,           # shapes loader only
        "splits": {"train": 0.8, "val": 0.1, "test": 0.1, "seed": 1},
        "augmentation": "natural_default",   # see AUGMENTATION_POLICIES
    },
    "model": {
        "fine_patch": 8,             # fine stream: textures / edges
        "coarse_patch": 16,          # coarse stream: geometry / context
        "dim": 96,                   # token dimension
        "depth": 3,                  # encoder layers L == latent depth Z
        "heads": 4,
        "mlp_ratio": 2.0,
        "cross_attention": "cls_bridged",   # cls_bridged (CrossViT) | full_pair (DSCATNet)
        "cross_rounds": 1,
        "volume_grid": 8,            # H' = W' of the latent volume
        "volume_channels": 24,       # C of the latent volume
        "som_grid": [4, 4, 4],       # 3-D SOM neuron grid
        "som_update": "gradient",    # gradient (DESOM-style) | kohonen_ema (Hebbian)
        "som_init": "data",          # data (sample first batch's voxels) | random
        "som_revival": True,         # re-seed dead neurons at each epoch end
        "proj_dim": 64,              # contrastive projection head output
    },
    "loss": {                        # standing defaults, DECISION-LOG
        "ntxent": 1.0, "som": 0.5, "smooth": 0.1, "order": 0.1, "geodesic": 0.0,
        "order_monotone": 0.05,      # per-slice centroid monotonicity penalty
        "tau_ntxent": 0.2, "tau_som": 0.5,
        "sigma_start": None, "sigma_end": None,  # null => grid-derived (max/2, 0.75)
        "smooth_axes": ["h", "w"],   # Z excluded (feedback); ["h","w","z"] = old behavior
        "order_fmax": 0.5,           # ordering regularizer max cutoff (cycles/px)
    },
    "train": {
        "batch_size": 32, "epochs": 3, "lr": 3e-4, "weight_decay": 0.05,
        "warmup_frac": 0.1, "amp": True, "num_workers": 2, "seed": 7,
        "som_sample_voxels": 2048,   # voxels sampled per step for the SOM
        "save_checkpoint": True,     # torch.save {model,som,config,history} post-train
        "checkpoint_dir": ".",       # Kaggle: set to "/kaggle/working"
    },
    "viz": {
        "probe_images": 6,           # images whose latent cubes we track
        "embed_subset": 400,         # points in the embedding animation
        "anim_fps": 6, "max_anim_frames": 60,
    },
}

PRESETS = {
    "ham10000": {   # Kaggle: kmader/skin-cancer-mnist-ham10000
        "dataset": {
            "name": "ham10000", "loader": "csv",
            "image_dir": [
                "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
                "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
            ],
            "metadata_csv": "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv",
            "path_column": "image_id", "path_suffix": ".jpg",
            "label_column": "dx", "group_column": "lesion_id",
            "image_size": 128, "n_per_class": None,
            "augmentation": "dermoscopy_default",
        },
        "model": {"dim": 256, "depth": 8, "heads": 8, "volume_grid": 16,
                  "volume_channels": 64, "som_grid": [6, 6, 6], "proj_dim": 128},
        "loss": {"sigma_start": None, "sigma_end": None},   # grid-derived anneal
        "train": {"batch_size": 128, "epochs": 30, "lr": 1e-3},
        "viz": {"probe_images": 8},
    },
    "eurosat": {    # Kaggle: apollo2506/eurosat-dataset
        "dataset": {
            "name": "eurosat", "loader": "imagefolder",
            "image_dir": "/kaggle/input/eurosat-dataset/EuroSAT",
            "label_column": "folder", "group_column": None,
            "image_size": 64, "n_per_class": None,
            "augmentation": "satellite_default",
        },
        "model": {"dim": 256, "depth": 8, "heads": 8, "volume_grid": 8,
                  "volume_channels": 64, "som_grid": [6, 6, 6], "proj_dim": 128},
        "loss": {"sigma_start": None, "sigma_end": None},   # grid-derived anneal
        "train": {"batch_size": 256, "epochs": 30, "lr": 1e-3},
        "viz": {"probe_images": 8},
    },
}

def apply_preset(name, base=None):
    """Deep-merge a preset over the base CONFIG (dataset swap = one line)."""
    cfg = copy.deepcopy(base or CONFIG)
    for section, overrides in PRESETS[name].items():
        cfg[section].update(overrides)
    return cfg

# CONFIG = apply_preset("ham10000")   # ◀ uncomment to swap dataset
# CONFIG = apply_preset("eurosat")    # ◀ uncomment to swap dataset


def validate_config(cfg):
    """Reject invalid configs with errors that name the offending field."""
    d, m = cfg["dataset"], cfg["model"]
    if d["loader"] not in ("shapes", "imagefolder", "csv"):
        raise ValueError(f"dataset.loader: unknown loader {d['loader']!r}")
    if d["image_size"] <= 0:
        raise ValueError("dataset.image_size: must be positive")
    if abs(sum(d["splits"][k] for k in ("train", "val", "test")) - 1.0) > 1e-6:
        raise ValueError("dataset.splits: train+val+test must sum to 1.0")
    for k in ("fine_patch", "coarse_patch"):
        if d["image_size"] % m[k]:
            raise ValueError(f"model.{k}: must divide dataset.image_size")
    if m["cross_attention"] not in ("cls_bridged", "full_pair"):
        raise ValueError(f"model.cross_attention: unknown mode {m['cross_attention']!r}")
    if m["som_update"] not in ("gradient", "kohonen_ema"):
        raise ValueError(f"model.som_update: unknown mode {m['som_update']!r}")
    if m.get("som_init", "data") not in ("data", "random"):
        raise ValueError(f"model.som_init: unknown mode {m.get('som_init')!r}")
    if not isinstance(m.get("som_revival", True), bool):
        raise ValueError("model.som_revival: must be a bool")
    if len(m["som_grid"]) != 3 or any(g <= 0 for g in m["som_grid"]):
        raise ValueError("model.som_grid: must be 3 positive ints")
    ls = cfg["loss"]
    for sk in ("sigma_start", "sigma_end"):
        if ls[sk] is not None and not (isinstance(ls[sk], (int, float))
                                       and not isinstance(ls[sk], bool) and ls[sk] > 0):
            raise ValueError(f"loss.{sk}: must be a positive number or null (grid-derived)")
    sa = ls.get("smooth_axes", ["h", "w"])
    if not sa or any(a not in ("h", "w", "z") for a in sa):
        raise ValueError("loss.smooth_axes: must be a non-empty subset of ['h','w','z']")
    if m["dim"] % m["heads"]:
        raise ValueError("model.dim: must be divisible by model.heads")
    if d["loader"] in ("imagefolder", "csv") and not d["image_dir"]:
        raise ValueError(f"dataset.image_dir: required for the {d['loader']} loader")
    if d["loader"] == "csv" and not d["metadata_csv"]:
        raise ValueError("dataset.metadata_csv: required for the csv loader")
    return cfg

CONFIG = validate_config(CONFIG)
print(f"dataset={CONFIG['dataset']['name']} · loader={CONFIG['dataset']['loader']} · "
      f"image={CONFIG['dataset']['image_size']}px · L={CONFIG['model']['depth']} · "
      f"volume={CONFIG['model']['volume_grid']}²×{CONFIG['model']['depth']}×{CONFIG['model']['volume_channels']} · "
      f"SOM={tuple(CONFIG['model']['som_grid'])}")

In [ ]:
# ── setup: imports · seeds · device ──────────────────────────────────────
import contextlib, csv as csvmod, hashlib, json, math, os, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "axes.grid": False})

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(CONFIG["train"]["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = bool(CONFIG["train"]["amp"] and DEVICE == "cuda")

def autocast_ctx():
    if USE_AMP:
        dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.autocast(DEVICE, dtype=dt)
    return contextlib.nullcontext()

print(f"device={DEVICE} · amp={USE_AMP} · torch={torch.__version__}")

## Stage 1 — Universal data pipeline

Three interchangeable loaders behind one interface (`items` = list of
`(source, label, group)`), a **named augmentation-policy registry** (medical
imagery must not get hue-destroying jitter — pick the policy in the config),
hash-based **grouped splits** (no `lesion_id` ever straddles train/test),
and a two-view contrastive wrapper.

In [ ]:
# ── data: loaders · augmentation registry · splits · two-view wrapper ────
SHAPE_CLASSES = ("circle", "square", "triangle")

def generate_shapes_image(rng, size, cls):
    """Deterministic synthetic image: one shape on a noisy background."""
    bg = (rng.standard_normal((size, size, 3)) * 18 + rng.integers(60, 190)) \
        .clip(0, 255).astype(np.uint8)
    img = Image.fromarray(bg)
    drw = ImageDraw.Draw(img)
    m = size // 4
    cx, cy = rng.integers(m, size - m, 2)
    r = int(rng.integers(size // 6, size // 3))
    color = tuple(int(c) for c in rng.integers(0, 255, 3))
    if cls == "circle":
        drw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    elif cls == "square":
        drw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=color)
    else:
        drw.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=color)
    return img

def build_items(cfg):
    """-> (items, class_names); items = [(source, label_idx_or_-1, group_key)]."""
    d = cfg["dataset"]
    if d["loader"] == "shapes":
        rng = np.random.default_rng(d["splits"]["seed"])
        items = [((cls, int(rng.integers(0, 2**31))), ci, None)
                 for ci, cls in enumerate(SHAPE_CLASSES)
                 for _ in range(d["n_per_class"])]
        return items, list(SHAPE_CLASSES)
    if d["loader"] == "imagefolder":
        root = Path(d["image_dir"])
        roots = [root / s for s in ("train", "val", "test") if (root / s).is_dir()] or [root]
        classes = sorted({p.name for r in roots for p in r.iterdir() if p.is_dir()})
        cidx = {c: i for i, c in enumerate(classes)}
        items = [(str(f), cidx[c.name] if d["label_column"] else -1, None)
                 for r in roots for c in sorted(r.iterdir()) if c.is_dir()
                 for f in sorted(c.iterdir())
                 if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif", ".tiff")]
        return items, classes
    # csv loader
    dirs = d["image_dir"] if isinstance(d["image_dir"], (list, tuple)) else [d["image_dir"]]
    with open(d["metadata_csv"], newline="") as fh:
        rows = list(csvmod.DictReader(fh))
    labels = sorted({r[d["label_column"]] for r in rows}) if d["label_column"] else []
    cidx = {c: i for i, c in enumerate(labels)}
    items = []
    for r in rows:
        name = r[d["path_column"]] + d["path_suffix"]
        path = next((os.path.join(dd, name) for dd in dirs
                     if os.path.exists(os.path.join(dd, name))), None)
        if path is None:
            continue
        items.append((path,
                      cidx[r[d["label_column"]]] if d["label_column"] else -1,
                      r[d["group_column"]] if d["group_column"] else None))
    return items, labels

def split_of(key, splits):
    """Stable hash -> train/val/test; same group key => same split, always."""
    h = int(hashlib.md5(f"{key}-{splits['seed']}".encode()).hexdigest(), 16) % 10**6 / 10**6
    if h < splits["train"]:
        return "train"
    return "val" if h < splits["train"] + splits["val"] else "test"

AUGMENTATION_POLICIES = {
    "none":               dict(),
    "natural_default":    dict(crop=(0.6, 1.0), hflip=True, rot=15,
                               bc=0.3, channel_jitter=0.10, noise=0.02),
    "dermoscopy_default": dict(crop=(0.7, 1.0), hflip=True, vflip=True, rot=180,
                               bc=0.2, channel_jitter=0.0, noise=0.01),  # no hue ops
    "satellite_default":  dict(crop=(0.6, 1.0), hflip=True, vflip=True, rot90=True,
                               bc=0.2, channel_jitter=0.05, noise=0.02),
}

def augment(img, size, p, rng):
    """PIL geometric ops, then tensor photometric ops. Returns [3,H,W] in [0,1]."""
    if p.get("crop"):
        s = rng.uniform(*p["crop"]); w = max(8, int(img.width * math.sqrt(s)))
        x0 = rng.integers(0, img.width - w + 1); y0 = rng.integers(0, img.height - w + 1)
        img = img.crop((x0, y0, x0 + w, y0 + w))
    if p.get("rot"):
        img = img.rotate(float(rng.uniform(-p["rot"], p["rot"])), resample=Image.BILINEAR)
    if p.get("rot90") and rng.random() < 0.75:
        img = img.rotate(90 * int(rng.integers(1, 4)))
    if p.get("hflip") and rng.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if p.get("vflip") and rng.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    img = img.resize((size, size), Image.BILINEAR)
    x = torch.from_numpy(np.asarray(img, np.float32) / 255.0).permute(2, 0, 1)
    if p.get("bc"):
        x = x * (1 + rng.uniform(-p["bc"], p["bc"])) + rng.uniform(-p["bc"], p["bc"]) * 0.5
    if p.get("channel_jitter"):
        x = x * (1 + torch.from_numpy(
            rng.uniform(-p["channel_jitter"], p["channel_jitter"], 3).astype(np.float32)
        ).view(3, 1, 1))
    if p.get("noise"):
        x = x + torch.randn_like(x) * p["noise"]
    return x.clamp(0, 1)

class UniversalDataset(Dataset):
    """One dataset class for all loaders; mode: 'two_view' (SSL) or 'eval'."""

    def __init__(self, cfg, split, mode):
        d = cfg["dataset"]
        all_items, self.classes = build_items(cfg)
        self.items = [it for i, it in enumerate(all_items)
                      if split_of(it[2] if it[2] is not None else i, d["splits"]) == split]
        self.size, self.mode = d["image_size"], mode
        self.policy = AUGMENTATION_POLICIES[d["augmentation"]]
        self.split = split

    def __len__(self):
        return len(self.items)

    def _load(self, source):
        if isinstance(source, tuple):  # shapes: (class_name, seed)
            cls, seed = source
            return generate_shapes_image(np.random.default_rng(seed), self.size * 2, cls)
        return Image.open(source).convert("RGB")

    def __getitem__(self, i):
        source, label, _ = self.items[i]
        img = self._load(source)
        if self.mode == "two_view":
            rng = np.random.default_rng()  # fresh randomness per view
            return (augment(img, self.size, self.policy, rng),
                    augment(img, self.size, self.policy, rng), label)
        x = torch.from_numpy(np.asarray(
            img.resize((self.size, self.size), Image.BILINEAR), np.float32
        ) / 255.0).permute(2, 0, 1)
        return x, label

In [ ]:
# ── instantiate data + look at what the model will see ───────────────────
train_ds = UniversalDataset(CONFIG, "train", "two_view")
eval_train_ds = UniversalDataset(CONFIG, "train", "eval")
eval_test_ds = UniversalDataset(CONFIG, "test", "eval")
CLASSES = train_ds.classes
NUM_CLASSES = len(CLASSES) if CONFIG["dataset"]["label_column"] else 0

train_loader = DataLoader(train_ds, batch_size=CONFIG["train"]["batch_size"],
                          shuffle=True, num_workers=CONFIG["train"]["num_workers"],
                          drop_last=len(train_ds) > CONFIG["train"]["batch_size"])
print(f"train={len(train_ds)} · eval-train={len(eval_train_ds)} · "
      f"eval-test={len(eval_test_ds)} · classes={CLASSES if NUM_CLASSES else 'unlabeled'}")

va, vb, yy = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
fig.suptitle(f"two augmented views per image — policy '{CONFIG['dataset']['augmentation']}'")
for j in range(6):
    for row, batch in enumerate((va, vb)):
        ax = axes[row, j]
        ax.imshow(batch[j].permute(1, 2, 0).numpy())
        ax.set_axis_off()
        if row == 0 and NUM_CLASSES:
            ax.set_title(CLASSES[yy[j]], fontsize=9)
plt.tight_layout(); plt.show()

## Stages 2–5 — Dual-scale backbone → 3-D latent volume

- **Dual-scale patch embedding**: a fine stream (textures, edges) and a
  coarse stream (geometry, context), each with its own CLS token.
- **Cross-scale attention**: `cls_bridged` (CrossViT — each stream's CLS
  queries the other stream; linear in tokens, the validated default) or
  `full_pair` (DSCATNet — every token attends across scales). Each round runs
  **cross-scale attention first, then per-stream self-attention**: in
  `cls_bridged` mode the cross step writes only to the CLS token, so self-attn
  must follow to spread that updated CLS into the patch tokens before fusion
  drops the CLS — otherwise the bridge never reaches the patches.
- **Fusion**: both streams resampled onto the common `H'×W'` grid, summed,
  projected — this grid *is* the spatial face of the latent volume.
- **Encoder**: pre-norm ViT; **every layer's output is retained**.
- **Spatial uplifting**: layer *l* → depth slice `V[:,:,l,:]`, giving
  `V ∈ R^{H'×W'×L×C}` — no global average pooling, no classifier.

In [ ]:
# ── model ────────────────────────────────────────────────────────────────
class SelfAttnBlock(nn.Module):
    """Pre-norm transformer block (self-attention + MLP)."""

    def __init__(self, dim, heads, mlp_ratio):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))

    def forward(self, x):
        a = self.n1(x)
        x = x + self.attn(a, a, a, need_weights=False)[0]
        return x + self.mlp(self.n2(x))

class CrossScaleBlock(nn.Module):
    """Information exchange between the fine and coarse token streams."""

    def __init__(self, dim, heads, mode):
        super().__init__()
        self.mode = mode
        self.nf, self.nc = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.f2c = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.c2f = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, xf, xc):
        f, c = self.nf(xf), self.nc(xc)
        if self.mode == "cls_bridged":   # CLS of one stream queries the other
            cf = xf[:, :1] + self.f2c(f[:, :1], c[:, 1:], c[:, 1:], need_weights=False)[0]
            cc = xc[:, :1] + self.c2f(c[:, :1], f[:, 1:], f[:, 1:], need_weights=False)[0]
            return (torch.cat([cf, xf[:, 1:]], 1), torch.cat([cc, xc[:, 1:]], 1))
        yf = xf + self.f2c(f, c, c, need_weights=False)[0]      # full_pair
        yc = xc + self.c2f(c, f, f, need_weights=False)[0]
        return yf, yc

class UMTViT(nn.Module):
    """Dual-scale encoder with spatial uplifting into a latent voxel volume.

    forward(x) -> dict:
      volume [B,H',W',L,Cv] · pooled [B,L*Cv] (per-slice means, label-free
      readout for probes) · proj [B,proj_dim] (contrastive head output)
    """

    def __init__(self, cfg):
        super().__init__()
        m, d = cfg["model"], cfg["dataset"]
        dim, L = m["dim"], m["depth"]
        self.vg, self.Cv, self.L = m["volume_grid"], m["volume_channels"], L
        self.gf = d["image_size"] // m["fine_patch"]
        self.gc = d["image_size"] // m["coarse_patch"]
        ch = d["channels"]
        self.embed_f = nn.Conv2d(ch, dim, m["fine_patch"], m["fine_patch"])
        self.embed_c = nn.Conv2d(ch, dim, m["coarse_patch"], m["coarse_patch"])
        self.cls_f = nn.Parameter(torch.zeros(1, 1, dim))
        self.cls_c = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_f = nn.Parameter(torch.randn(1, self.gf**2 + 1, dim) * 0.02)
        self.pos_c = nn.Parameter(torch.randn(1, self.gc**2 + 1, dim) * 0.02)
        self.cross = nn.ModuleList(
            CrossScaleBlock(dim, m["heads"], m["cross_attention"])
            for _ in range(m["cross_rounds"]))
        self.stream_f = nn.ModuleList(
            SelfAttnBlock(dim, m["heads"], m["mlp_ratio"]) for _ in range(m["cross_rounds"]))
        self.stream_c = nn.ModuleList(
            SelfAttnBlock(dim, m["heads"], m["mlp_ratio"]) for _ in range(m["cross_rounds"]))
        self.fuse = nn.Linear(dim, dim)
        self.pos_fused = nn.Parameter(torch.randn(1, self.vg**2, dim) * 0.02)
        self.encoder = nn.ModuleList(
            SelfAttnBlock(dim, m["heads"], m["mlp_ratio"]) for _ in range(L))
        self.uplift = nn.ModuleList(nn.Linear(dim, self.Cv) for _ in range(L))
        self.head = nn.Sequential(nn.Linear(L * self.Cv, dim), nn.GELU(),
                                  nn.Linear(dim, m["proj_dim"]))

    def _to_grid(self, tokens, g):
        """[B,g*g,D] tokens -> [B,D,vg,vg] resampled onto the fusion grid."""
        B, _, D = tokens.shape
        x = tokens.transpose(1, 2).reshape(B, D, g, g)
        if g != self.vg:
            x = F.interpolate(x, size=(self.vg, self.vg),
                              mode="bilinear", align_corners=False)
        return x

    def forward(self, x):
        B = x.shape[0]
        xf = self.embed_f(x).flatten(2).transpose(1, 2)
        xc = self.embed_c(x).flatten(2).transpose(1, 2)
        xf = torch.cat([self.cls_f.expand(B, -1, -1), xf], 1) + self.pos_f
        xc = torch.cat([self.cls_c.expand(B, -1, -1), xc], 1) + self.pos_c
        for cross, sf, sc in zip(self.cross, self.stream_f, self.stream_c):
            # Cross first, then per-stream self-attn spreads the updated CLS
            # into the patch tokens before fusion drops the CLS.
            xf, xc = cross(xf, xc)
            xf, xc = sf(xf), sc(xc)
        grid = (self._to_grid(xf[:, 1:], self.gf)
                + self._to_grid(xc[:, 1:], self.gc))     # [B,D,vg,vg]
        tokens = grid.flatten(2).transpose(1, 2)         # [B,vg*vg,D]
        t = self.fuse(tokens) + self.pos_fused
        slices = []
        for blk, lift in zip(self.encoder, self.uplift):
            t = blk(t)
            slices.append(lift(t).reshape(B, self.vg, self.vg, self.Cv))
        volume = torch.stack(slices, dim=3)              # [B,H',W',L,Cv]
        pooled = volume.mean(dim=(1, 2)).flatten(1)      # [B,L*Cv]
        return {"volume": volume, "pooled": pooled, "proj": self.head(pooled)}

model = UMTViT(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    probe_out = model(va[:2].to(DEVICE))
print(f"UMT-ViT: {n_params/1e6:.2f} M params · volume {tuple(probe_out['volume'].shape[1:])} "
      f"· pooled {probe_out['pooled'].shape[1]} dims · proj {probe_out['proj'].shape[1]} dims")

## Stage 6 — Differentiable 3-D Self-Organizing Map

Every voxel feature is softly assigned to neurons on a 3-D grid; the BMU's
Gaussian neighborhood (σ annealed over training, DESOM schedule) pulls
nearby neurons toward similar patterns — *"neurons that activate together
become more similar"*, made differentiable. `som_update="kohonen_ema"`
switches to the literal Kohonen/Hebbian rule (updates outside autograd)
for the biological ablation.

In [ ]:
# ── 3-D SOM ──────────────────────────────────────────────────────────────
class Soft3DSOM(nn.Module):
    """Topology-preserving 3-D SOM over voxel features, trainable end-to-end."""

    def __init__(self, grid, feat_dim, tau, update="gradient", ema_lr=0.5):
        super().__init__()
        self.grid, self.K = tuple(grid), int(np.prod(grid))
        self.tau, self.update, self.ema_lr = tau, update, ema_lr
        w = torch.randn(self.K, feat_dim) * 0.5
        self.weights = nn.Parameter(w, requires_grad=(update == "gradient"))
        zz, yy, xx = torch.meshgrid(*(torch.arange(g).float() for g in grid),
                                    indexing="ij")
        coords = torch.stack([zz, yy, xx], -1).reshape(self.K, 3)
        self.register_buffer("grid_d2", torch.cdist(coords, coords) ** 2)
        self.register_buffer("coords", coords)

    def bmu(self, v):
        return torch.cdist(v, self.weights.detach()).argmin(1)

    @torch.no_grad()
    def data_init(self, pool, noise=0.05):
        """Seed every neuron from a random voxel of `pool` (+ scaled noise).
        Data-driven init (model.som_init='data') breaks the collapse-to-mean
        failure of random init under a wide neighborhood σ."""
        if pool.shape[0] == 0:
            return
        idx = torch.randint(0, pool.shape[0], (self.K,), device=pool.device)
        src = pool[idx]
        self.weights.data.copy_(src + torch.randn_like(src) * noise * (pool.std() + 1e-8))

    @torch.no_grad()
    def revive(self, hit_counts, pool, noise=0.1):
        """Re-seed neurons that won zero BMU assignments this epoch to a random
        recently-seen voxel (+ noise). Returns the number revived."""
        if pool.shape[0] == 0:
            return 0
        dead = torch.nonzero(hit_counts == 0, as_tuple=False).flatten()
        if dead.numel() == 0:
            return 0
        src = pool[torch.randint(0, pool.shape[0], (dead.numel(),), device=pool.device)]
        self.weights.data[dead] = src + torch.randn_like(src) * noise * (pool.std() + 1e-8)
        return int(dead.numel())

    def loss(self, v, sigma):
        """Neighborhood-weighted quantization loss (+ EMA update if Hebbian)."""
        d2 = torch.cdist(v, self.weights) ** 2                    # [M,K]
        bmu = d2.detach().argmin(1)
        h = torch.exp(-self.grid_d2[bmu] / (2 * sigma**2))        # [M,K]
        if self.update == "kohonen_ema":
            with torch.no_grad():                                  # Hebbian step
                num = h.T @ v.detach()
                den = h.sum(0).unsqueeze(1) + 1e-8
                self.weights.lerp_(num / den, self.ema_lr)
            return (v - self.weights[bmu].detach()).pow(2).sum(1).mean()
        return (h * d2).sum(1).div(h.sum(1) + 1e-8).mean()

    @torch.no_grad()
    def metrics(self, v):
        """quantization error · topographic error · dead-neuron fraction."""
        d = torch.cdist(v, self.weights)
        near = d.topk(2, largest=False).indices
        qe = d.gather(1, near[:, :1]).mean().item()
        te = (self.grid_d2[near[:, 0], near[:, 1]] > 3.0 + 1e-6).float().mean().item()
        dead = 1.0 - torch.unique(near[:, 0]).numel() / self.K
        return {"quantization_error": qe, "topographic_error": te,
                "dead_neuron_fraction": dead}

som = Soft3DSOM(CONFIG["model"]["som_grid"], CONFIG["model"]["volume_channels"],
                CONFIG["loss"]["tau_som"], CONFIG["model"]["som_update"]).to(DEVICE)
print(f"SOM: grid {som.grid} = {som.K} neurons · update={som.update}")

## Stage 8 — Self-supervised objectives

`L = λ₁·NT-Xent + λ₂·SOM + λ₃·smoothness + λ₄·ordering (+ λ₅·geodesic)`

- **NT-Xent** — two views of one image stay close in projection space.
- **SOM quantization** — voxels move toward their (neighborhood-weighted) BMU.
- **Smoothness** — total variation across the volume's 3-D neighbor graph.
- **Ordering** — the imposed Z-axis bias: slice *l* is penalized for spatial
  power above cutoff `f_max·(1−l/L)`, pushing deep slices toward coarse
  structure. Whether ordering *emerges* is measured later, not assumed.
- **Geodesic** — the proposal's most speculative term (see RESEARCH §6);
  shipped behind `loss.geodesic > 0`, off by default.

In [ ]:
# ── losses ───────────────────────────────────────────────────────────────
def nt_xent(za, zb, tau):
    """SimCLR NT-Xent over a batch of paired views."""
    z = F.normalize(torch.cat([za, zb]), dim=1)
    sim = z @ z.T / tau
    n = za.shape[0]
    sim.fill_diagonal_(float("-inf"))
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

_SMOOTH_AXIS = {"h": 1, "w": 2, "z": 3}   # V is [B,H',W',L,Cv]

def smoothness_loss(V, axes=("h", "w")):
    """TV over the selected volume neighbor edges (normalized per edge).
    Z is excluded by default — TV along depth is an antagonist to depth
    differentiation (feedback); pass ['h','w','z'] to restore the old term."""
    terms = [V.diff(dim=_SMOOTH_AXIS[a]).pow(2).mean() for a in axes]
    return sum(terms) / max(len(terms), 1)

def _radius_grid(h, w, device):
    fy = torch.fft.fftfreq(h, device=device).abs()
    fx = torch.fft.rfftfreq(w, device=device)
    return torch.sqrt(fy[:, None] ** 2 + fx[None, :] ** 2)   # cycles/px

def monotone_centroid_loss(V):
    """Differentiable per-slice spectral centroid, penalizing any rise with
    depth: Σ_l relu(c[l+1] − c[l]). Reuses the FFT/radius machinery of
    ordering_loss so the constraint acts directly on the centroid (not just an
    upper envelope of high-frequency power)."""
    B, H, W, L, C = V.shape
    spec = torch.fft.rfft2(V.permute(0, 3, 4, 1, 2), norm="ortho").abs() ** 2   # [B,L,C,H,Wf]
    r = _radius_grid(H, W, V.device)                                            # [H,Wf]
    cs = [(spec[:, l] * r).sum() / (spec[:, l].sum() + 1e-8) for l in range(L)]
    total = V.new_zeros(())
    for l in range(L - 1):
        total = total + F.relu(cs[l + 1] - cs[l])
    return total

def ordering_loss(V, fmax):
    """Penalize per-slice spatial power above a depth-decreasing cutoff."""
    B, H, W, L, C = V.shape
    spec = torch.fft.rfft2(V.permute(0, 3, 4, 1, 2), norm="ortho").abs() ** 2
    r = _radius_grid(H, W, V.device)                          # [H, W//2+1]
    total = V.new_zeros(())
    for l in range(L):
        cutoff = fmax * (1 - l / max(L - 1, 1))
        mask = (r > cutoff).float()
        total = total + (spec[:, l] * mask).mean()
    return total / L

def geodesic_loss(v_sub, za, zb, k=6):
    """Batch k-NN-graph shortest path between the two views' embeddings.
    Gradients flow through edge lengths on the (detached) chosen path.
    Ablation-gated: only called when loss.geodesic > 0."""
    import heapq
    nodes = torch.cat([F.normalize(za, dim=1).mean(0, keepdim=True),
                       F.normalize(zb, dim=1).mean(0, keepdim=True),
                       F.normalize(v_sub[:256, : za.shape[1]], dim=1)])
    d = torch.cdist(nodes, nodes)
    knn = d.detach().topk(min(k + 1, len(nodes)), largest=False).indices[:, 1:]
    dist = {0: 0.0}; prev = {}; pq = [(0.0, 0)]; seen = set()
    while pq:
        du, u = heapq.heappop(pq)
        if u in seen:
            continue
        seen.add(u)
        if u == 1:
            break
        for vtx in knn[u].tolist():
            alt = du + d[u, vtx].item()
            if alt < dist.get(vtx, float("inf")):
                dist[vtx] = alt; prev[vtx] = u; heapq.heappush(pq, (alt, vtx))
    if 1 not in prev:
        return d[0, 1]
    loss, node = d.new_zeros(()), 1
    while node != 0:
        loss, node = loss + d[prev[node], node], prev[node]
    return loss

## Training — self-supervised, label-free

AdamW + cosine schedule with warmup, AMP on GPU, SOM neighborhood σ
annealed exponentially (DESOM schedule). After every epoch we snapshot
the SOM weights, the probe images' latent cubes, and an embedding subset —
these snapshots power the animations below.

In [ ]:
# ── training loop ────────────────────────────────────────────────────────
from torch.utils.data import Subset

L_CFG, T_CFG, V_CFG = CONFIG["loss"], CONFIG["train"], CONFIG["viz"]

params = list(model.parameters()) + (
    list(som.parameters()) if som.update == "gradient" else [])
opt = torch.optim.AdamW(params, lr=T_CFG["lr"], weight_decay=T_CFG["weight_decay"])
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

total_steps = max(1, T_CFG["epochs"] * len(train_loader))
warmup = max(1, int(total_steps * T_CFG["warmup_frac"]))

def lr_at(step):
    if step < warmup:
        return step / warmup
    p = (step - warmup) / max(1, total_steps - warmup)
    return 0.5 * (1 + math.cos(math.pi * p))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)

# grid-derived σ defaults when loss.sigma_start/end are null: a start of
# max(grid)/2 keeps the initial neighborhood inside the grid (a σ wider than
# the grid pulls every neuron to the global mean and the map never
# differentiates); explicit numbers in the config are honored as-is.
SIGMA_START = (L_CFG["sigma_start"] if L_CFG["sigma_start"] is not None
               else max(CONFIG["model"]["som_grid"]) / 2)
SIGMA_END = L_CFG["sigma_end"] if L_CFG["sigma_end"] is not None else 0.75
print(f"SOM σ anneal: {SIGMA_START:.2f} → {SIGMA_END:.2f} "
      f"({'grid-derived' if L_CFG['sigma_start'] is None else 'config'} start)")

def sigma_at(step):
    f = step / total_steps
    return SIGMA_START * (SIGMA_END / SIGMA_START) ** f

# fixed probe images (latent-cube tracking) + fixed embedding subset
probe_ds = eval_test_ds if len(eval_test_ds) >= V_CFG["probe_images"] else eval_train_ds
probe_x = torch.stack([probe_ds[i][0] for i in range(V_CFG["probe_images"])]).to(DEVICE)
probe_y = [probe_ds[i][1] for i in range(V_CFG["probe_images"])]
# seeded random subset (NOT first-N: a dataset's CSV ordering can make the
# leading rows nearly single-class, which starves the embedding legend)
_embed_rng = np.random.default_rng(T_CFG["seed"])
_n_embed = min(V_CFG["embed_subset"], len(eval_train_ds))
_embed_idx = sorted(_embed_rng.permutation(len(eval_train_ds))[:_n_embed].tolist())
embed_ds = Subset(eval_train_ds, _embed_idx)
embed_loader = DataLoader(embed_ds, batch_size=64)

@torch.no_grad()
def snapshot(epoch):
    model.eval()
    vols = model(probe_x)["volume"].to(torch.float16).cpu().numpy()
    feats, labels = [], []
    for x, y in embed_loader:
        feats.append(model(x.to(DEVICE))["pooled"].cpu())
        labels.append(y)
    feats = torch.cat(feats)
    vox = torch.from_numpy(vols.astype(np.float32)).reshape(-1, som.weights.shape[1])
    snap = {
        "epoch": epoch,
        "som_weights": som.weights.detach().cpu().numpy().copy(),
        "probe_volumes": vols,
        "embeddings": feats.numpy(),
        "labels": torch.cat(labels).numpy(),
        "som_metrics": som.metrics(vox.to(DEVICE)),
    }
    model.train()
    return snap

history = {k: [] for k in ("total", "ntxent", "som", "smooth", "order",
                           "order_monotone", "geodesic", "sigma", "lr")}
snapshots = [snapshot(0)]   # untrained state — animations start here
model.train()
step = 0
t0 = time.time()
for epoch in range(1, T_CFG["epochs"] + 1):
    epoch_hits = torch.zeros(som.K, device=DEVICE)   # BMU wins this epoch
    recent_vox = None                                # revival source pool
    for xa, xb, _ in train_loader:
        xa, xb = xa.to(DEVICE), xb.to(DEVICE)
        sigma = sigma_at(step)
        with autocast_ctx():
            oa, ob = model(xa), model(xb)
            V = oa["volume"]
            vox = V.reshape(-1, V.shape[-1])
            idx = torch.randperm(vox.shape[0], device=DEVICE)[:T_CFG["som_sample_voxels"]]
            v_sub = vox[idx]
            v_sub_f = v_sub.float()
            # data-driven SOM init on the very first batch's voxels
            if step == 0 and CONFIG["model"]["som_init"] == "data":
                som.data_init(v_sub_f.detach())
            terms = {
                "ntxent": nt_xent(oa["proj"], ob["proj"], L_CFG["tau_ntxent"]),
                "som":    som.loss(v_sub_f, sigma),
                "smooth": smoothness_loss(V, L_CFG["smooth_axes"]),
                "order":  ordering_loss(V, L_CFG["order_fmax"]),
                "order_monotone": monotone_centroid_loss(V),
            }
            terms["geodesic"] = (
                geodesic_loss(v_sub_f, oa["proj"].float(), ob["proj"].float())
                if L_CFG["geodesic"] > 0 else V.new_zeros(()))
            loss = sum(L_CFG[k] * terms[k] for k in terms)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        with torch.no_grad():
            epoch_hits += torch.bincount(som.bmu(v_sub_f.detach()), minlength=som.K).float()
            recent_vox = v_sub_f.detach()
        for k, t in terms.items():
            history[k].append(float(t.detach()))
        history["total"].append(float(loss.detach()))
        history["sigma"].append(sigma)
        history["lr"].append(opt.param_groups[0]["lr"])
        step += 1
    # dead-neuron revival: re-seed this epoch's non-winners from recent voxels
    revived = (som.revive(epoch_hits, recent_vox)
               if CONFIG["model"]["som_revival"] and recent_vox is not None else 0)
    snapshots.append(snapshot(epoch))
    m = snapshots[-1]["som_metrics"]
    print(f"epoch {epoch:>3}/{T_CFG['epochs']} · loss {history['total'][-1]:.3f} "
          f"(ntxent {history['ntxent'][-1]:.3f} · som {history['som'][-1]:.3f}) · "
          f"QE {m['quantization_error']:.3f} · TE {m['topographic_error']:.3f} · "
          f"dead {m['dead_neuron_fraction']:.2f} · revived {revived:>3} · "
          f"σ {history['sigma'][-1]:.2f}")
print(f"trained {step} steps in {time.time()-t0:.0f}s on {DEVICE}")

## Checkpoint

Persist the trained weights so a later session can resume or run eval-only
without retraining (the owner did this by hand on Kaggle — formalized here).
Set `train.checkpoint_dir` to `/kaggle/working` on Kaggle so the file lands
in the writable output area. To reload, uncomment the snippet in the next cell.

In [ ]:
# ── save checkpoint ────────────────────────────────────────────────────────
if T_CFG.get("save_checkpoint", True):
    ckpt_dir = Path(T_CFG.get("checkpoint_dir", "."))
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"umtvit_{CONFIG['dataset']['name']}.pt"
    torch.save({"model": model.state_dict(), "som": som.state_dict(),
                "config": CONFIG, "history": history}, ckpt_path)
    print(f"checkpoint saved · {ckpt_path} · {ckpt_path.stat().st_size/1024:.0f} KB")
else:
    print("checkpointing disabled (train.save_checkpoint=False)")

In [ ]:
# ── resume / eval-only (uncomment in a fresh session) ──────────────────────
# ckpt = torch.load(
#     Path(CONFIG["train"]["checkpoint_dir"]) / f"umtvit_{CONFIG['dataset']['name']}.pt",
#     map_location=DEVICE)
# model.load_state_dict(ckpt["model"]); som.load_state_dict(ckpt["som"])
# history = ckpt["history"]
# model.eval()
# print("resumed:", ckpt["config"]["dataset"]["name"],
#       "· epochs trained:", len(history["total"]) // max(1, len(train_loader)))

In [ ]:
# ── training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for k in ("total", "ntxent", "som"):
    axes[0].plot(history[k], label=k)
axes[0].set_title("main losses"); axes[0].set_xlabel("step"); axes[0].legend()
for k in ("smooth", "order", "order_monotone"):
    axes[1].plot(history[k], label=k)
axes[1].set_title("topology regularizers"); axes[1].set_xlabel("step"); axes[1].legend()
ep = [s["epoch"] for s in snapshots]
for key, style in (("quantization_error", "-o"), ("topographic_error", "-s"),
                   ("dead_neuron_fraction", "-^")):
    axes[2].plot(ep, [s["som_metrics"][key] for s in snapshots], style,
                 label=key.replace("_", " "))
axes[2].set_title("SOM organization over training"); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## The latent cube — scroll through learned depth

Each column is one encoder layer's slice of the voxel volume (channel-mean,
per-slice normalized). The Z-axis is the *learned* hierarchy: with the
ordering regularizer, shallow slices should stay detailed and deep slices
should smooth out. The animation glides through the depth axis.

In [ ]:
# ── latent-cube slice montage ────────────────────────────────────────────
def slice_img(vol, l):
    """Channel-mean of depth slice l, per-slice min-max normalized."""
    s = vol[:, :, l, :].astype(np.float32).mean(-1)
    return (s - s.min()) / (np.ptp(s) + 1e-8)

final_vols = snapshots[-1]["probe_volumes"]
n_show = min(3, final_vols.shape[0])
L = final_vols.shape[3]
fig, axes = plt.subplots(n_show, L + 1, figsize=(1.6 * (L + 1), 1.8 * n_show),
                         squeeze=False)
for i in range(n_show):
    axes[i, 0].imshow(probe_x[i].permute(1, 2, 0).cpu().numpy())
    axes[i, 0].set_ylabel(CLASSES[probe_y[i]] if NUM_CLASSES else f"img {i}",
                          fontsize=8)
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    for l in range(L):
        axes[i, l + 1].imshow(slice_img(final_vols[i], l), cmap="magma")
        axes[i, l + 1].set_axis_off()
        if i == 0:
            axes[i, l + 1].set_title(f"z={l}", fontsize=9)
fig.suptitle("latent voxel volume — depth slices (z = encoder layer)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── animation 1: gliding through the latent cube's depth axis ────────────
def cube_animation(vol, image, fps, max_frames):
    interp = max(2, min(10, max_frames // max(L - 1, 1)))
    zs = np.linspace(0, L - 1, (L - 1) * interp + 1)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(6.4, 3.2))
    ax0.imshow(image); ax0.set_title("input", fontsize=9); ax0.set_axis_off()
    im = ax1.imshow(slice_img(vol, 0), cmap="magma", animated=True)
    ax1.set_axis_off()
    title = ax1.set_title("z = 0.0", fontsize=9)

    def frame(z):
        lo, hi = int(np.floor(z)), min(int(np.floor(z)) + 1, L - 1)
        a = z - np.floor(z)
        im.set_array((1 - a) * slice_img(vol, lo) + a * slice_img(vol, hi))
        title.set_text(f"latent depth z = {z:.1f} / {L - 1}")
        return [im, title]

    anim = animation.FuncAnimation(fig, frame, frames=zs, interval=1000 / fps,
                                   blit=False)
    plt.close(fig)
    return anim

anim1 = cube_animation(final_vols[0], probe_x[0].permute(1, 2, 0).cpu().numpy(),
                       V_CFG["anim_fps"], V_CFG["max_anim_frames"])
display(HTML(anim1.to_jshtml()))

## The SOM — a topographic atlas of visual structure

**U-matrix**: mean feature-distance of each neuron to its 3-D grid
neighbors (dark valleys = coherent regions, bright walls = cluster
boundaries). **Hit maps**: where the eval images' voxels land. The
animation replays the map organizing itself over training.

In [ ]:
# ── SOM maps (final state) ───────────────────────────────────────────────
GZ, GY, GX = som.grid

def umatrix_layers(weights):
    """[K,C] SOM weights -> U-matrix per z-layer of the neuron grid."""
    w = weights.reshape(GZ, GY, GX, -1)
    u = np.zeros((GZ, GY, GX))
    for z in range(GZ):
        for y in range(GY):
            for x in range(GX):
                nb = [w[c] for c in ((z + dz, y + dy, x + dx)
                      for dz, dy, dx in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)))
                      if 0 <= c[0] < GZ and 0 <= c[1] < GY and 0 <= c[2] < GX]
                u[z, y, x] = np.mean([np.linalg.norm(w[z, y, x] - n) for n in nb])
    return u

@torch.no_grad()
def bmu_hits():
    vox = torch.from_numpy(
        snapshots[-1]["probe_volumes"].astype(np.float32)
    ).reshape(-1, som.weights.shape[1]).to(DEVICE)
    counts = torch.bincount(som.bmu(vox), minlength=som.K).cpu().numpy()
    return counts.reshape(GZ, GY, GX)

u_final, hits = umatrix_layers(snapshots[-1]["som_weights"]), bmu_hits()
fig, axes = plt.subplots(2, GZ, figsize=(2.1 * GZ, 4.4), squeeze=False)
for z in range(GZ):
    axes[0, z].imshow(u_final[z], cmap="viridis")
    axes[0, z].set_title(f"SOM z-layer {z}", fontsize=9); axes[0, z].set_axis_off()
    axes[1, z].imshow(hits[z], cmap="hot")
    axes[1, z].set_axis_off()
axes[0, 0].set_ylabel("U-matrix"); axes[1, 0].set_ylabel("hits")
fig.suptitle("3-D SOM — U-matrix (top) and voxel hit maps (bottom) per grid layer")
plt.tight_layout(); plt.show()

In [ ]:
# ── animation 2: the SOM organizing itself over training ─────────────────
u_stack = [umatrix_layers(s["som_weights"]) for s in snapshots]
vmax = max(u.max() for u in u_stack) + 1e-8
fig, axes = plt.subplots(1, GZ, figsize=(2.1 * GZ, 2.5), squeeze=False)
ims = []
for z in range(GZ):
    ims.append(axes[0, z].imshow(u_stack[0][z], cmap="viridis", vmin=0, vmax=vmax,
                                 animated=True))
    axes[0, z].set_axis_off()
sup = fig.suptitle("SOM U-matrix — epoch 0")

def som_frame(i):
    for z in range(GZ):
        ims[z].set_array(u_stack[i][z])
    sup.set_text(f"SOM U-matrix — epoch {snapshots[i]['epoch']}"
                 f" (σ={sigma_at(min(snapshots[i]['epoch']*len(train_loader), total_steps-1)):.2f})")
    return ims

anim2 = animation.FuncAnimation(fig, som_frame, frames=len(snapshots),
                                interval=700, blit=False)
plt.close(fig)
display(HTML(anim2.to_jshtml()))

In [ ]:
# ── animation 3: the embedding space taking shape ────────────────────────
last = snapshots[-1]["embeddings"]
mu = last.mean(0)
_, _, Vt = np.linalg.svd(last - mu, full_matrices=False)   # fixed PCA basis
proj_all = [(s["embeddings"] - mu) @ Vt[:2].T for s in snapshots]
lims = np.abs(np.concatenate(proj_all)).max() * 1.1
lab = snapshots[-1]["labels"]
# explicit per-point tab10 colors (RGBA) so the legend, built from CLASSES,
# always matches — no legend_elements handle/label mismatch when the sampled
# subset happens to miss a class
from matplotlib.lines import Line2D
tab10 = plt.cm.tab10
pt_colors = (tab10(np.asarray(lab) % 10) if NUM_CLASSES
             else np.tile(tab10(0), (len(lab), 1)))

fig, ax = plt.subplots(figsize=(5.2, 4.6))
sc = ax.scatter(*proj_all[0].T, c=pt_colors, s=14, alpha=0.8)
ax.set_xlim(-lims, lims); ax.set_ylim(-lims, lims)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ttl = ax.set_title("pooled embeddings — epoch 0")
if NUM_CLASSES:
    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      markerfacecolor=tab10(i % 10), markeredgecolor="none",
                      label=CLASSES[i]) for i in range(NUM_CLASSES)]
    ax.legend(handles=handles, fontsize=8, loc="upper right")

def emb_frame(i):
    sc.set_offsets(proj_all[i])
    ttl.set_text(f"pooled embeddings (fixed PCA basis) — epoch {snapshots[i]['epoch']}")
    return [sc, ttl]

anim3 = animation.FuncAnimation(fig, emb_frame, frames=len(snapshots),
                                interval=700, blit=False)
plt.close(fig)
display(HTML(anim3.to_jshtml()))

## Did the Z-axis actually order by scale? (measured, not assumed)

Radially averaged spatial power spectra per depth slice, plus each slice's
spectral centroid. If the ordering regularizer did its job, the centroid
frequency should **fall with depth**. If it doesn't, that is a reportable
negative result — see the honesty notes at the top.

In [ ]:
# ── Z-axis frequency probe ───────────────────────────────────────────────
def radial_spectrum(img2d):
    f = np.abs(np.fft.rfft2(img2d - img2d.mean(), norm="ortho")) ** 2
    fy = np.abs(np.fft.fftfreq(img2d.shape[0]))
    fx = np.fft.rfftfreq(img2d.shape[1])
    r = np.sqrt(fy[:, None] ** 2 + fx[None, :] ** 2)
    bins = np.linspace(0, r.max() + 1e-9, 9)
    idx = np.digitize(r.ravel(), bins) - 1
    power = np.bincount(idx, weights=f.ravel(), minlength=len(bins)) / \
        (np.bincount(idx, minlength=len(bins)) + 1e-9)
    centers = (bins[:-1] + bins[1:]) / 2
    return centers, power[: len(centers)]

# The fair measurement (feedback §2.3): take the spectral centroid of EACH
# channel, then average — a channel-mean image can cancel per-channel high-
# frequency structure and understate the true centroid. The old channel-mean
# centroid is kept alongside, once, for comparability. `centroids` (the fair
# one) drives the chart, the export, and the report.
C_ch = final_vols.shape[4]
spectra, centroids, centroids_channel_mean = [], [], []
for l in range(L):
    ps_cm = [radial_spectrum(slice_img(final_vols[i], l))
             for i in range(final_vols.shape[0])]
    centers = ps_cm[0][0]
    mean_p = np.mean([p for _, p in ps_cm], axis=0)
    spectra.append(mean_p)                                    # channel-mean spectra (display)
    centroids_channel_mean.append((centers * mean_p).sum() / (mean_p.sum() + 1e-9))
    per_ch = []
    for i in range(final_vols.shape[0]):
        for c in range(C_ch):
            cen, p = radial_spectrum(final_vols[i][:, :, l, c].astype(np.float32))
            per_ch.append((cen * p).sum() / (p.sum() + 1e-9))
    centroids.append(float(np.mean(per_ch)))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 3.6))
for l, p in enumerate(spectra):
    ax0.plot(centers, p / (p.sum() + 1e-9), label=f"z={l}",
             color=plt.cm.plasma(l / max(L - 1, 1)))
ax0.set_xlabel("spatial frequency (cycles/px)"); ax0.set_ylabel("normalized power")
ax0.set_title("per-slice power spectra (channel-mean)"); ax0.legend(fontsize=8)
ax1.bar(range(L), centroids, color=plt.cm.plasma(np.linspace(0, 1, L)))
ax1.set_xlabel("depth slice z (encoder layer)")
ax1.set_ylabel("mean per-channel spectral centroid")
trend = "decreases with depth ✔ (ordering emerged)" \
    if all(np.diff(centroids) <= 1e-4) else \
    "is NOT monotone — ordering only partially emerged (honest negative result)"
ax1.set_title(f"spectral centroid {trend}", fontsize=9)
plt.tight_layout(); plt.show()
print("spectral centroids by depth (mean per-channel):", np.round(centroids, 4))
print("spectral centroids by depth (channel-mean, old):", np.round(centroids_channel_mean, 4))

## Evaluation — labels enter here only

Everything above ran **without labels**. The representation was learned
purely from the self-supervised objectives (contrastive + SOM + smoothness
+ ordering). Only now, to *measure* what was learned, do labels enter — and
only as frozen-feature yardsticks, never as a training signal to the model.

- **Linear probe**: a single logistic-regression layer on frozen pooled
  features (the standard SSL probe). Compared against chance = 1/`NUM_CLASSES`.
- **k-NN (k=5, cosine)**: non-parametric readout of the same features.
- **SOM quality**: quantization error, topographic error, dead-neuron rate on
  held-out test voxels.
- **Trustworthiness (k=7)**: does the pooled feature space preserve the local
  neighbor structure of raw input pixels?

These are **SSL-probe numbers on a tiny CPU smoke run** — they are *not*
comparable to supervised end-to-end training (e.g. DSCATNet's 97.8% on
HAM10000). They exist to confirm the pipeline learns *something* structured
and to give the export a scorecard. Unlabeled mode skips probe/k-NN.

In [ ]:
# ── feature extraction over eval splits (frozen, no_grad, batched) ────────
@torch.no_grad()
def extract_features(ds, max_n=None):
    """-> (pooled [N,D], labels [N], flat_pixels [N,3*S*S]) for an eval dataset."""
    model.eval()
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    feats, labels, pixels, seen = [], [], [], 0
    for x, y in loader:
        feats.append(model(x.to(DEVICE))["pooled"].cpu())
        labels.append(y)
        pixels.append(x.reshape(x.shape[0], -1))
        seen += x.shape[0]
        if max_n is not None and seen >= max_n:
            break
    if not feats:
        D = CONFIG["model"]["depth"] * CONFIG["model"]["volume_channels"]
        return (torch.zeros(0, D), torch.zeros(0, dtype=torch.long),
                torch.zeros(0, 3 * ds.size * ds.size))
    F_ = torch.cat(feats); Y_ = torch.cat(labels); P_ = torch.cat(pixels)
    if max_n is not None:
        F_, Y_, P_ = F_[:max_n], Y_[:max_n], P_[:max_n]
    return F_, Y_, P_

Xtr, ytr, _        = extract_features(eval_train_ds)
Xte, yte, Pte      = extract_features(eval_test_ds)
print(f"features: train {tuple(Xtr.shape)} · test {tuple(Xte.shape)} · "
      f"NUM_CLASSES={NUM_CLASSES}")

In [ ]:
# ── linear probe (pure-torch logistic regression) + k-NN (k=5, cosine) ────
results = {}

def standardize(train, test):
    mu, sd = train.mean(0, keepdim=True), train.std(0, keepdim=True) + 1e-6
    return (train - mu) / sd, (test - mu) / sd

if NUM_CLASSES >= 2 and len(Xtr) > 0 and len(Xte) > 0:
    Xtr_s, Xte_s = standardize(Xtr, Xte)
    chance = 1.0 / NUM_CLASSES

    # linear probe: one Linear layer, ~400 AdamW steps, full-batch
    probe = nn.Linear(Xtr_s.shape[1], NUM_CLASSES)
    popt = torch.optim.AdamW(probe.parameters(), lr=1e-2, weight_decay=1e-4)
    Xtr_d, ytr_d = Xtr_s.to(DEVICE), ytr.to(DEVICE)
    probe.to(DEVICE)
    for _ in range(400):
        popt.zero_grad(set_to_none=True)
        F.cross_entropy(probe(Xtr_d), ytr_d).backward()
        popt.step()
    with torch.no_grad():
        pred = probe(Xte_s.to(DEVICE)).argmax(1).cpu()
    probe_acc = (pred == yte).float().mean().item()

    # k-NN, k=5, cosine similarity
    def knn_accuracy(Xtr_, ytr_, Xte_, yte_, k=5):
        a = F.normalize(Xtr_, dim=1); b = F.normalize(Xte_, dim=1)
        sim = b @ a.T                                   # [Nte, Ntr]
        k = min(k, a.shape[0])
        nn_idx = sim.topk(k, dim=1).indices             # [Nte, k]
        votes = ytr_[nn_idx]                            # [Nte, k]
        pred = torch.mode(votes, dim=1).values
        return (pred == yte_).float().mean().item()

    knn_acc = knn_accuracy(Xtr_s, ytr, Xte_s, yte, k=5)
    results.update(linear_probe_acc=probe_acc, knn_acc=knn_acc, chance=chance)
    print(f"linear probe: {probe_acc:.3f}   k-NN(5,cos): {knn_acc:.3f}   "
          f"chance: {chance:.3f}   (test n={len(yte)})")
else:
    results.update(linear_probe_acc=None, knn_acc=None, chance=None)
    print("unlabeled mode (NUM_CLASSES<2): linear probe / k-NN skipped — "
          "SSL training and topology metrics are unaffected.")

In [ ]:
# ── SOM quality on held-out test voxels + trustworthiness ─────────────────
@torch.no_grad()
def test_voxels(ds, max_imgs=64):
    model.eval()
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    vox, seen = [], 0
    for x, _ in loader:
        V = model(x.to(DEVICE))["volume"]               # [B,H,W,L,Cv]
        vox.append(V.reshape(-1, V.shape[-1]).cpu())
        seen += x.shape[0]
        if seen >= max_imgs:
            break
    return torch.cat(vox) if vox else torch.zeros(0, som.weights.shape[1])

vox_test = test_voxels(eval_test_ds)
if len(vox_test) > som.K:
    sub = vox_test[torch.randperm(len(vox_test))[:CONFIG["train"]["som_sample_voxels"]]]
    som_eval = som.metrics(sub.to(DEVICE))
else:
    som_eval = som.metrics(vox_test.to(DEVICE)) if len(vox_test) else {
        "quantization_error": float("nan"), "topographic_error": float("nan"),
        "dead_neuron_fraction": float("nan")}
results["som_metrics"] = som_eval


def trustworthiness(X_high, X_low, k=7):
    """Local-neighbor preservation from input pixels (high) to features (low).
    Standard formula (Venna & Kaski); pure numpy on <=400 samples."""
    X_high = np.asarray(X_high, np.float32); X_low = np.asarray(X_low, np.float32)
    n = X_high.shape[0]
    k = int(min(k, (n - 1) // 2))
    if n < 4 or k < 1:
        return float("nan")

    def pdist(A):
        g = (A * A).sum(1)
        d2 = g[:, None] + g[None, :] - 2.0 * (A @ A.T)
        return np.sqrt(np.maximum(d2, 0.0))

    Dh, Dl = pdist(X_high), pdist(X_low)
    ind_high = np.argsort(Dh, axis=1)
    rank_high = np.argsort(ind_high, axis=1)            # rank of j from i (0=self)
    nn_low = np.argsort(Dl, axis=1)[:, 1:k + 1]         # k nearest in feature space
    t = 0.0
    for i in range(n):
        for j in nn_low[i]:
            r = rank_high[i, j]
            if r > k:
                t += r - k
    norm = n * k * (2 * n - 3 * k - 1)
    return float(1.0 - (2.0 / norm) * t) if norm > 0 else float("nan")

n_tw = min(400, len(Xte))
tw = trustworthiness(Pte[:n_tw].numpy(), Xte[:n_tw].numpy(), k=7) if n_tw >= 4 \
    else float("nan")
results["trustworthiness_k7"] = tw

In [ ]:
# ── results table (honest framing) ────────────────────────────────────────
print("=" * 60)
print("  UMT-ViT evaluation — frozen-feature SSL probes")
print("=" * 60)
lp = results["linear_probe_acc"]; kn = results["knn_acc"]; ch = results["chance"]
if lp is not None:
    print(f"  linear probe accuracy    {lp:6.3f}   (chance {ch:.3f})")
    print(f"  k-NN (k=5, cosine)       {kn:6.3f}   (chance {ch:.3f})")
else:
    print("  linear probe / k-NN      —  (unlabeled mode)")
print(f"  SOM quantization error   {som_eval['quantization_error']:6.3f}")
print(f"  SOM topographic error    {som_eval['topographic_error']:6.3f}   (lower=better)")
print(f"  SOM dead-neuron fraction {som_eval['dead_neuron_fraction']:6.3f}")
print(f"  trustworthiness (k=7)    {tw:6.3f}   (1.0=perfect neighbor preservation)")
print("=" * 60)
_is_smoke = DEVICE == "cpu" or CONFIG["train"]["epochs"] <= 5
if _is_smoke:
    print(f"NOTE: these are SSL-probe numbers on a short smoke run ({DEVICE}, "
          f"{CONFIG['train']['epochs']} epochs).")
    print("They are NOT comparable to supervised end-to-end training. Scale up")
    print("(preset + GPU + more epochs) before drawing conclusions about")
    print("representation quality; here they only confirm the pipeline learns")
    print("structured features.")
else:
    print("NOTE: these are frozen-feature SSL yardsticks — a linear probe and")
    print("k-NN read out features learned WITHOUT labels. They are not directly")
    print("comparable to supervised end-to-end results (e.g. DSCATNet's 97.8% on")
    print("HAM10000); they measure representation quality on the common SSL scale.")

## Export artifacts

Self-describing, re-loadable outputs (fp16 `.npz` + JSON sidecars, same
discipline as the ViTreous packs). Everything a downstream viewer or the
ablation matrix needs: the probe images' latent cubes, the SOM weights
(initial and final), the full training history with per-epoch SOM metrics,
and a human-readable run report.

In [ ]:
# ── write artifacts ───────────────────────────────────────────────────────
art_dir = Path("umtvit_artifacts") / CONFIG["dataset"]["name"]
art_dir.mkdir(parents=True, exist_ok=True)

# 1) latent cubes of the probe images (fp16) + JSON sidecar
cubes = snapshots[-1]["probe_volumes"].astype(np.float16)   # [N,H',W',L,Cv]
np.savez_compressed(art_dir / "latent_cubes.npz", cubes=cubes,
                    labels=np.asarray(probe_y))
sidecar = {
    "array": "cubes",
    "dtype": "float16",
    "shape": list(cubes.shape),
    "axes": ["image", "H_prime", "W_prime", "Z", "C"],
    "axis_meaning": {
        "H_prime": "latent volume height (fusion grid)",
        "W_prime": "latent volume width (fusion grid)",
        "Z": "transformer depth == encoder layer index; a LEARNED hierarchy "
             "of representations, NOT physical/anatomical depth",
        "C": "per-slice latent channels (volume_channels)",
    },
    "labels": [int(v) for v in probe_y],
    "class_names": list(CLASSES) if NUM_CLASSES else None,
    "dataset": CONFIG["dataset"]["name"],
}
(art_dir / "latent_cubes.json").write_text(json.dumps(sidecar, indent=2))

# 2) SOM weights — final + initial (fp16)
np.savez_compressed(
    art_dir / "som_weights.npz",
    final=snapshots[-1]["som_weights"].astype(np.float16),
    initial=snapshots[0]["som_weights"].astype(np.float16),
    grid=np.asarray(som.grid))

# 3) full training history + per-epoch SOM metrics
run_history = {
    "config": CONFIG,
    "history": {k: [float(x) for x in v] for k, v in history.items()},
    "per_epoch_som_metrics": [
        {"epoch": int(s["epoch"]),
         **{k: float(v) for k, v in s["som_metrics"].items()}}
        for s in snapshots],
    "eval": {
        "linear_probe_acc": results["linear_probe_acc"],
        "knn_acc": results["knn_acc"],
        "chance": results["chance"],
        "som_metrics": {k: float(v) for k, v in som_eval.items()},
        "trustworthiness_k7": float(tw),
    },
    "spectral_centroids_by_depth": [float(c) for c in centroids],
    "spectral_centroids_channel_mean": [float(c) for c in centroids_channel_mean],
}
(art_dir / "run_history.json").write_text(json.dumps(run_history, indent=2))

# 4) human-readable report
def _fmt(v):
    return "—" if v is None else f"{v:.4f}"

report = f"""# UMT-ViT run report — {CONFIG['dataset']['name']}

Self-supervised, dataset-agnostic run. Labels (if any) were used **only** for
the evaluation probes below, never for representation learning.

## Configuration
- dataset: `{CONFIG['dataset']['name']}` · loader: `{CONFIG['dataset']['loader']}` ·
  image {CONFIG['dataset']['image_size']}px · classes: {NUM_CLASSES if NUM_CLASSES else 'unlabeled'}
- model: dim {CONFIG['model']['dim']} · depth/Z {CONFIG['model']['depth']} ·
  volume {CONFIG['model']['volume_grid']}²×{CONFIG['model']['depth']}×{CONFIG['model']['volume_channels']} ·
  SOM {tuple(CONFIG['model']['som_grid'])} · cross-attn `{CONFIG['model']['cross_attention']}` ·
  SOM update `{CONFIG['model']['som_update']}`
- training: {CONFIG['train']['epochs']} epochs · batch {CONFIG['train']['batch_size']} ·
  lr {CONFIG['train']['lr']}

## Loss weights
{', '.join(f'{k}={CONFIG["loss"][k]}' for k in ('ntxent','som','smooth','order','order_monotone','geodesic'))}
smooth_axes={CONFIG['loss']['smooth_axes']}

## Final loss values
{', '.join(f'{k}={history[k][-1]:.4f}' for k in ('total','ntxent','som','smooth','order','order_monotone') if history[k])}

## Evaluation (frozen-feature SSL probes — NOT supervised parity)
| metric | value |
|---|---|
| linear probe accuracy | {_fmt(results['linear_probe_acc'])} |
| k-NN (k=5, cosine) | {_fmt(results['knn_acc'])} |
| chance (1/classes) | {_fmt(results['chance'])} |
| SOM quantization error | {som_eval['quantization_error']:.4f} |
| SOM topographic error | {som_eval['topographic_error']:.4f} |
| SOM dead-neuron fraction | {som_eval['dead_neuron_fraction']:.4f} |
| trustworthiness (k=7) | {tw:.4f} |

## Z-axis ordering (measured, not assumed)
Per-slice spectral centroids by depth (cycles/px):
`{[round(float(c), 4) for c in centroids]}`

A centroid that **falls with depth** indicates the imposed ordering bias took
effect (shallow slices carry fine detail, deep slices coarse structure).
Whether it is monotone here is a measured, reportable outcome — see the
frequency-probe cell and the honesty notes at the top of the notebook. The
Z-axis is a learned representational hierarchy, **not** physical depth.

## Artifacts
- `latent_cubes.npz` (+ `.json` sidecar) — probe-image latent volumes (fp16)
- `som_weights.npz` — initial + final SOM weights and grid shape
- `run_history.json` — config, full loss history, per-epoch SOM metrics, eval
"""
(art_dir / "report.md").write_text(report)

# 5) file listing with sizes
print(f"artifacts written to {art_dir}/")
for f in sorted(art_dir.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size/1024:8.1f} KB")

## Export web bundle

One self-contained `umtvit_web.json` for the **UMT-ViT Explorer** web app
(`apps/web`, the `/umtvit` route) — the same run, explorable interactively in
the browser: the latent-cube depth scrubber, the SOM self-organization
replay, the embedding-formation replay, training curves, and the Z-axis
honesty panel. Schema version 1; every field derives from objects already in
scope (CONFIG, history, snapshots, results, centroids). Downsampled and
rounded to stay well under ~4 MB at notebook defaults.

In [ ]:
# ── build umtvit_web.json (web explorer bundle, schema v1) ────────────────
import base64, io

def _finite(v):
    """JSON-safe float: None for NaN/inf (schema allows nulls for metrics)."""
    if v is None:
        return None
    v = float(v)
    return v if math.isfinite(v) else None

def _downsample_idx(n, cap=400):
    """Evenly-spaced <=cap indices over range(n), endpoints preserved."""
    if n <= cap:
        return list(range(n))
    return np.unique(np.linspace(0, n - 1, cap).round().astype(int)).tolist()

def _png_b64(chw, max_px=128):
    """[3,H,W] in [0,1] -> base64 PNG string, longest side <= max_px."""
    arr = (chw.clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    im = Image.fromarray(arr)
    if max(im.size) > max_px:
        scale = max_px / max(im.size)
        im = im.resize((max(1, round(im.width * scale)),
                        max(1, round(im.height * scale))), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

def _cube(vol):
    """[H',W',L,Cv] volume -> [L][H'][W'] channel-mean, per-slice norm, 3dp."""
    return [np.round(slice_img(vol, l), 3).tolist() for l in range(vol.shape[2])]

# history: one shared downsample index set across all series
_hidx = _downsample_idx(len(history["total"]))
_web_series = {
    k: ([float(history[k][i]) for i in _hidx] if history[k] else [])
    for k in ("total", "ntxent", "som", "smooth", "order")
}

# probes: input thumbnail + initial and final latent cubes
init_vols = snapshots[0]["probe_volumes"]
web_probes = [
    {
        "label": (CLASSES[probe_y[i]] if (NUM_CLASSES and probe_y[i] >= 0) else None),
        "image_png_b64": _png_b64(probe_x[i]),
        "cube_initial": _cube(init_vols[i]),
        "cube_final": _cube(final_vols[i]),
    }
    for i in range(final_vols.shape[0])
]

web_bundle = {
    "version": 1,
    "dataset": {
        "name": CONFIG["dataset"]["name"],
        "image_size": CONFIG["dataset"]["image_size"],
        "augmentation": CONFIG["dataset"]["augmentation"],
        "num_classes": int(NUM_CLASSES),
    },
    "model": {
        "dim": CONFIG["model"]["dim"],
        "depth": CONFIG["model"]["depth"],
        "volume_grid": CONFIG["model"]["volume_grid"],
        "volume_channels": CONFIG["model"]["volume_channels"],
        "som_grid": list(CONFIG["model"]["som_grid"]),
        "cross_attention": CONFIG["model"]["cross_attention"],
        "params_millions": round(n_params / 1e6, 3),
    },
    "metrics": {
        "linear_probe": _finite(results["linear_probe_acc"]),
        "knn": _finite(results["knn_acc"]),
        "chance": _finite(results["chance"]),
        "som_quantization_error": _finite(som_eval["quantization_error"]),
        "som_topographic_error": _finite(som_eval["topographic_error"]),
        "som_dead_fraction": _finite(som_eval["dead_neuron_fraction"]),
        "trustworthiness": _finite(tw),
    },
    "spectral_centroids": [_finite(c) for c in centroids],
    "classes": list(CLASSES) if NUM_CLASSES else None,
    "history": {"steps": [int(i) for i in _hidx], "series": _web_series},
    "probes": web_probes,
    "som": {
        "grid": [int(GZ), int(GY), int(GX)],
        "epochs": [int(s["epoch"]) for s in snapshots],
        "umatrix": [np.round(u, 3).tolist() for u in u_stack],
        "hits_final": hits.astype(int).tolist(),
    },
    "embeddings": {
        "epochs": [int(s["epoch"]) for s in snapshots],
        "coords": [np.round(p, 2).tolist() for p in proj_all],
        "labels": ([int(v) for v in lab] if NUM_CLASSES else None),
    },
}

web_path = art_dir / "umtvit_web.json"
web_path.write_text(json.dumps(web_bundle, separators=(",", ":")))
_kb = web_path.stat().st_size / 1024
print(f"wrote {web_path} · {_kb:.1f} KB ({_kb/1024:.2f} MB)")
print("drop this file onto the /umtvit page of the web app")

## Conclusion

**What this notebook demonstrated.** A single config cell drove a complete,
label-free pipeline end to end: universal data loading (shapes here; swap to
`ham10000`/`eurosat` by uncommenting one preset line), a dual-scale
cross-attention ViT whose every layer is *uplifted* into a 3-D latent voxel
volume, a differentiable 3-D SOM that reorganizes that volume into a
topology-preserving atlas, and four self-supervised objectives — with the
whole latent geometry made directly inspectable (depth-slice montages, the
gliding-cube animation, SOM U-matrix/hit maps, the SOM-organizing and
embedding-forming animations). Labels entered only at the evaluation step.

**The Z-axis result — honestly.** ViT layers do not order themselves by
spatial scale (Raghu et al., NeurIPS 2021); we *imposed* an ordering bias
and then **measured** the outcome. Read the spectral-centroid bar chart and
the printed `spectral centroids by depth`: a monotone fall with depth means
the bias took hold; a non-monotone profile is a genuine, reportable negative
result, not a bug. On this short CPU smoke run do not over-read either way —
the measurement is the deliverable, the verdict needs the scaled-up run.

**Where to read more.** Design contract and math:
[`docs/UMT-VIT-ARCHITECTURE.md`](https://github.com/LarsGroep/DragonHatchling/blob/claude/umt-vit-opus-orchestration-zpd03a/docs/UMT-VIT-ARCHITECTURE.md)
(§3 objectives, §6 evaluation, §8 artifacts); research grounding and the
critical evaluation of the proposal:
[`docs/UMT-VIT-RESEARCH.md`](https://github.com/LarsGroep/DragonHatchling/blob/claude/umt-vit-opus-orchestration-zpd03a/docs/UMT-VIT-RESEARCH.md).

**Scaling up.** This ran on CPU at a deliberately tiny scale. To do real
science: uncomment `CONFIG = apply_preset("ham10000")` (or `"eurosat"`) in
the config cell, run on a Kaggle T4/P100 GPU (AMP switches on automatically),
and raise `epochs`. The exported artifacts are dataset-tagged, so runs never
clobber each other, and the SSL probe numbers only become meaningful — and
only then loosely comparable to a supervised reference — after a full run.